In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

Cloning into 'CTAB-GAN-Plus'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 77 (delta 21), reused 17 (delta 17), pack-reused 48 (from 1)
Receiving objects: 100% (77/77), 1.05 MiB | 14.21 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [2]:
pip install ucimlrepo

In [3]:
pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 6.5 MB/s eta 0:00:00


In [4]:
from ucimlrepo import fetch_ucirepo

In [5]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sdv.metadata import SingleTableMetadata
import torch
import random

air_quality = fetch_ucirepo(id=360)
X = air_quality.data.features.copy()
print(air_quality.metadata)
print(air_quality.variables)

target_col = "CO(GT)"
X = X.drop(columns=["Date", "Time"], errors="ignore")
data = X.copy()
for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
data = data.replace(-200, np.nan)
for col in data.columns:
    data[col] = data[col].fillna(data[col].median())
data = data.dropna().reset_index(drop=True)

# Drop date/time/session/ID features before generator training (high cardinality).
_drop_feature_cols = [
    "Date", "Time", "date_time",
    "session_id", "Session ID", "Session_ID", "session",
    "X1 transaction date",
]
data = data.drop(columns=[c for c in _drop_feature_cols if c in data.columns], errors="ignore")

n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data[col] = data[col].fillna(data[col].median())

processed_data = data.copy()

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


{'uci_id': 360, 'name': 'Air Quality', 'repository_url': 'https://archive.ics.uci.edu/dataset/360/air+quality', 'data_url': 'https://archive.ics.uci.edu/static/public/360/data.csv', 'abstract': 'Contains the responses of a gas multisensor device deployed on the field in an Italian city. Hourly responses averages are recorded along with gas concentrations references from a certified analyzer. ', 'area': 'Computer Science', 'tasks': ['Regression'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 9358, 'num_features': 15, 'feature_types': ['Real'], 'demographics': [], 'target_col': None, 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2008, 'last_updated': 'Sun Mar 10 2024', 'dataset_doi': '10.24432/C59K5F', 'creators': ['Saverio Vito'], 'intro_paper': {'ID': 420, 'type': 'NATIVE', 'title': 'On field calibration of an electronic nose for benzene estimation in an urban pollution monitoring scenario', 'authors': 

In [6]:
from sklearn.model_selection import train_test_split
from sdv.evaluation.single_table import evaluate_quality

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
        random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

try:
    data_path = "air_quality_train.csv"
    train_real.to_csv(data_path, index=False)

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=[],
        log_columns=[],
        mixed_columns={},
        integer_columns=[],
        problem_type={"Regression": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    # Convert all columns back to numeric where possible
    for col in synthetic_ctabgan.columns:
        synthetic_ctabgan[col] = pd.to_numeric(
            synthetic_ctabgan[col],
            errors="coerce"
        )

        synthetic_ctabgan[col] = synthetic_ctabgan[col].fillna(
            train_real[col].median()
        )

    # Air Quality quality target is an integer score between 0 and 10
    pass  # keep continuous regression target as-is

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)



================ SINGLE RUN ================


100%|██████████| 150/150 [00:58<00:00,  2.57it/s]


Finished training in 68.38795375823975  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 242.90it/s]|
Column Shapes Score: 85.11%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 141.55it/s]|
Column Pair Trends Score: 78.47%

Overall Score (Average): 81.79%

CTABGAN: 0.8179


In [8]:
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

# WGAN-GP

try:

    import traceback

    data_wgan = train_real.copy()

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 310.69it/s]|
Column Shapes Score: 86.96%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 202.93it/s]|
Column Pair Trends Score: 98.02%

Overall Score (Average): 92.49%

WGAN_GP: 0.9249


In [9]:
# SDV MODELS

from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        pass  # keep continuous regression target as-is

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 250.36it/s]|
Column Shapes Score: 78.39%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 189.81it/s]|
Column Pair Trends Score: 64.35%

Overall Score (Average): 71.37%

CTGAN: 0.7137
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 213.47it/s]|
Column Shapes Score: 71.8%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 201.92it/s]|
Column Pair Trends Score: 65.74%

Overall Score (Average): 68.77%

CopulaGAN: 0.6877
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 349.54it/s]|
Column Shapes Score: 89.04%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:00<00:00, 194.94it/s]|
Column Pair Trends Score: 94.96%

Overall Score (Average): 92.0%

TVAE: 0.92
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 382.87it/s]|
Column Shapes Scor

In [15]:
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

regressors = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    'SVR_RBF': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoost': GradientBoostingRegressor(random_state=42),
}

# Changed EVAL_SEEDS to include multiple seeds for meaningful standard deviation calculation
EVAL_SEEDS = [SEED, SEED + 1, SEED + 2, SEED + 3, SEED + 4]
GENERATORS_TO_EVAL = list(synthetic_datasets.keys())

print(f'Regression evaluation: {len(regressors)} models, {len(EVAL_SEEDS)} seeds, {len(GENERATORS_TO_EVAL)} generators')


Regression evaluation: 10 models, 5 seeds, 6 generators


In [12]:
def evaluate_regression_models(train_df, test_df, label_col, models, test_size=0.2, seeds=None, use_holdout=False, schema_df=None):
    if seeds is None:
        seeds = EVAL_SEEDS
    if schema_df is None:
        schema_df = test_df if use_holdout else train_df

    train_df = align_to_train_schema(train_df, schema_df, label_col)
    test_df = align_to_train_schema(test_df, schema_df, label_col)
    results = []

    for name, model in models.items():
        r2_scores = []
        mse_scores = []
        rmse_scores = []
        mae_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if not use_holdout:
                X_train, _, y_train, _ = train_test_split(
                    X_train, y_train, test_size=test_size, random_state=seed
                )
                _, X_test, _, y_test = train_test_split(
                    X_test, y_test, test_size=test_size, random_state=seed
                )

            reg = clone(model)
            if 'random_state' in reg.get_params():
                reg.set_params(random_state=seed)

            reg.fit(X_train, y_train)
            y_pred = reg.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            r2_scores.append(r2_score(y_test, y_pred))
            mse_scores.append(mse)
            rmse_scores.append(np.sqrt(mse))
            mae_scores.append(mean_absolute_error(y_test, y_pred))

        results.append({
            'Model': name,
            'R2 Mean': np.mean(r2_scores),
            'R2 Std': np.std(r2_scores),
            'MSE Mean': np.mean(mse_scores),
            'MSE Std': np.std(mse_scores),
            'RMSE Mean': np.mean(rmse_scores),
            'RMSE Std': np.std(rmse_scores),
            'MAE Mean': np.mean(mae_scores),
            'MAE Std': np.std(mae_scores),
            'R2 (Mean±Std)': f"{np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}",
            'MSE (Mean±Std)': f"{np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}",
            'RMSE (Mean±Std)': f"{np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}",
            'MAE (Mean±Std)': f"{np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by='R2 Mean', ascending=False)


In [17]:
import pandas as pd
import numpy as np

def align_to_train_schema(df, schema_df, label_col):
    schema_columns = list(schema_df.columns)

    # Drop columns in df not present in schema_df
    extra_cols_in_df = [col for col in df.columns if col not in schema_columns]
    df = df.drop(columns=extra_cols_in_df, errors='ignore')

    # Add columns from schema_df not present in df, fill with NaN
    missing_cols_in_df = [col for col in schema_columns if col not in df.columns]
    for col in missing_cols_in_df:
        df[col] = np.nan # Use NaN, assuming numerical data for regression.

    # Reorder columns to match schema_df
    df = df[schema_columns]
    return df

print('TRTR (Train Real, Test Real) — 80% train / 20% holdout')
trtr_results = evaluate_regression_models(
    train_df=processed_data, # Changed to use full processed_data
    test_df=processed_data,   # Changed to use full processed_data
    label_col=target_col,
    models=regressors,
    seeds=EVAL_SEEDS,
    use_holdout=False,        # Changed to False to allow internal splitting per seed
    schema_df=processed_data, # Changed schema_df to processed_data
)
display(trtr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

all_comparisons = []
for synth_name in GENERATORS_TO_EVAL:
    if synth_name not in synthetic_datasets:
        print(f'Skipping {synth_name} - no synthetic dataset')
        continue

    print(f'{synth_name} - TSTR (train on synthetic, test on 20% holdout)')
    tstr_results = evaluate_regression_models(
        train_df=synthetic_datasets[synth_name],
        test_df=test_real,
        label_col=target_col,
        models=regressors,
        seeds=EVAL_SEEDS,
        use_holdout=False,        # Changed to False to allow internal splitting of synthetic data and test_real per seed
        schema_df=train_real,
    )
    display(tstr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

    comparison = trtr_results.merge(tstr_results, on='Model', suffixes=('_TRTR', '_TSTR'))
    comparison['R2_Drop'] = comparison['R2 Mean_TRTR'] - comparison['R2 Mean_TSTR']
    comparison['MSE_Increase'] = comparison['MSE Mean_TSTR'] - comparison['MSE Mean_TRTR']
    comparison['RMSE_Increase'] = comparison['RMSE Mean_TSTR'] - comparison['RMSE Mean_TRTR']
    comparison['MAE_Increase'] = comparison['MAE Mean_TSTR'] - comparison['MAE Mean_TRTR']
    comparison['Synthetic_Model'] = synth_name
    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)
summary = (
    combined_comparison
    .groupby('Synthetic_Model', as_index=False)[['R2_Drop', 'MSE_Increase', 'RMSE_Increase', 'MAE_Increase']]
    .mean()
    .sort_values('R2_Drop')
)

display(summary)


TRTR (Train Real, Test Real) — 80% train / 20% holdout


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
8,ExtraTrees,0.8838 ± 0.0083,0.2224 ± 0.0467,0.4690 ± 0.0494,0.3220 ± 0.0243
9,GradientBoost,0.8740 ± 0.0134,0.2393 ± 0.0427,0.4871 ± 0.0444,0.3445 ± 0.0272
7,RandomForest,0.8693 ± 0.0295,0.2490 ± 0.0669,0.4942 ± 0.0684,0.3335 ± 0.0332
0,LinearRegression,0.8540 ± 0.0168,0.2756 ± 0.0426,0.5234 ± 0.0409,0.3601 ± 0.0275
1,Ridge,0.8539 ± 0.0169,0.2757 ± 0.0426,0.5235 ± 0.0409,0.3601 ± 0.0274
3,ElasticNet,0.8537 ± 0.0169,0.2760 ± 0.0425,0.5238 ± 0.0407,0.3603 ± 0.0273
2,Lasso,0.8535 ± 0.0170,0.2763 ± 0.0424,0.5241 ± 0.0406,0.3605 ± 0.0270
4,SVR_RBF,0.8359 ± 0.0195,0.3094 ± 0.0458,0.5547 ± 0.0420,0.3618 ± 0.0314
5,KNN,0.8229 ± 0.0118,0.3352 ± 0.0477,0.5775 ± 0.0418,0.4020 ± 0.0231
6,DecisionTree,0.7446 ± 0.0589,0.4749 ± 0.0894,0.6862 ± 0.0634,0.4414 ± 0.0279


CTABGAN - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
2,Lasso,0.4672 ± 0.0640,0.7376 ± 0.2258,0.8497 ± 0.1252,0.6699 ± 0.0481
3,ElasticNet,0.4669 ± 0.0641,0.7380 ± 0.2259,0.8499 ± 0.1252,0.6700 ± 0.0481
1,Ridge,0.4667 ± 0.0641,0.7383 ± 0.2259,0.8501 ± 0.1252,0.6701 ± 0.0482
0,LinearRegression,0.4665 ± 0.0642,0.7385 ± 0.2259,0.8502 ± 0.1252,0.6702 ± 0.0482
4,SVR_RBF,0.4457 ± 0.0103,0.8049 ± 0.3607,0.8781 ± 0.1837,0.6442 ± 0.0666
8,ExtraTrees,0.3168 ± 0.2119,0.9123 ± 0.2688,0.9454 ± 0.1361,0.7169 ± 0.0837
7,RandomForest,0.1415 ± 0.1936,1.1375 ± 0.2272,1.0617 ± 0.1017,0.7879 ± 0.0888
5,KNN,-0.0763 ± 0.2349,1.4322 ± 0.3051,1.1905 ± 0.1219,0.9423 ± 0.0609
9,GradientBoost,-0.0999 ± 0.4055,1.4819 ± 0.5966,1.1925 ± 0.2446,0.8783 ± 0.1513
6,DecisionTree,-1.0604 ± 0.7046,2.6860 ± 0.6535,1.6268 ± 0.1991,1.1563 ± 0.1777


WGAN_GP - TSTR (train on synthetic, test on 20% holdout)


...

TVAE - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
4,SVR_RBF,0.7529 ± 0.0625,0.3346 ± 0.1066,0.5723 ± 0.0844,0.3939 ± 0.0463
8,ExtraTrees,0.7438 ± 0.0658,0.3385 ± 0.0728,0.5786 ± 0.0610,0.4178 ± 0.0493
7,RandomForest,0.7406 ± 0.0575,0.3433 ± 0.0631,0.5834 ± 0.0539,0.4203 ± 0.0405
9,GradientBoost,0.7262 ± 0.0443,0.3748 ± 0.1029,0.6066 ± 0.0822,0.4216 ± 0.0583
5,KNN,0.6614 ± 0.1381,0.4431 ± 0.1392,0.6579 ± 0.1011,0.4891 ± 0.0603
6,DecisionTree,0.3374 ± 0.3002,0.8501 ± 0.2902,0.9076 ± 0.1622,0.6595 ± 0.0947
2,Lasso,-0.5448 ± 2.6183,2.1130 ± 3.6145,1.0511 ± 1.0041,0.4979 ± 0.2457
3,ElasticNet,-0.5605 ± 2.6497,2.1346 ± 3.6578,1.0546 ± 1.0111,0.4985 ± 0.2471
1,Ridge,-0.5759 ± 2.6806,2.1558 ± 3.7003,1.0580 ± 1.0180,0.4992 ± 0.2486
0,LinearRegression,-0.5780 ± 2.6847,2.1586 ± 3.7059,1.0585 ± 1.0189,0.4992 ± 0.2488


GaussianCopula - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
0,LinearRegression,0.7924 ± 0.0897,0.2533 ± 0.0384,0.5017 ± 0.0399,0.3552 ± 0.0265
1,Ridge,0.7923 ± 0.0894,0.2535 ± 0.0382,0.5020 ± 0.0396,0.3554 ± 0.0265
3,ElasticNet,0.7922 ± 0.0893,0.2538 ± 0.0381,0.5022 ± 0.0395,0.3556 ± 0.0265
2,Lasso,0.7920 ± 0.0890,0.2540 ± 0.0379,0.5025 ± 0.0393,0.3559 ± 0.0264
4,SVR_RBF,0.7886 ± 0.0840,0.2675 ± 0.0507,0.5147 ± 0.0508,0.3583 ± 0.0378
9,GradientBoost,0.7710 ± 0.1213,0.2719 ± 0.0577,0.5187 ± 0.0536,0.3555 ± 0.0305
7,RandomForest,0.7707 ± 0.0950,0.2809 ± 0.0304,0.5292 ± 0.0289,0.3651 ± 0.0346
8,ExtraTrees,0.7674 ± 0.1052,0.2841 ± 0.0434,0.5314 ± 0.0413,0.3612 ± 0.0325
5,KNN,0.6983 ± 0.1031,0.3780 ± 0.0227,0.6145 ± 0.0186,0.4600 ± 0.0349
6,DecisionTree,0.5638 ± 0.1556,0.5463 ± 0.0681,0.7376 ± 0.0469,0.5685 ± 0.0366


,Synthetic_Model,R2_Drop,MSE_Increase,RMSE_Increase,MAE_Increase
3,GaussianCopula,0.091699,0.010940,0.009094,0.024458
5,WGAN_GP,0.108271,0.041328,0.039619,0.070220
4,TVAE,0.674256,0.831256,0.276521,0.115084
0,CTABGAN,0.691095,0.847316,0.493122,0.416004
2,CopulaGAN,0.839034,1.118909,0.597442,0.424818
1,CTGAN,0.884279,1.195546,0.647412,0.469447


In [18]:
output_file = 'TRTR_TSTR_results_air_quality.xlsx'

# Create quality_df from the scores dictionary
quality_df = pd.DataFrame(scores.items(), columns=['Model', 'Quality Score'])

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    quality_df.to_excel(writer, sheet_name='Quality_Metrics', index=False)
    trtr_results.to_excel(writer, sheet_name='TRTR_Results', index=False)
    combined_comparison.to_excel(writer, sheet_name='All_Comparisons', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    for synth_name in GENERATORS_TO_EVAL:
        if synth_name in combined_comparison['Synthetic_Model'].values:
            synth_results = combined_comparison[combined_comparison['Synthetic_Model'] == synth_name]
            synth_results.to_excel(writer, sheet_name=synth_name[:31], index=False)

print(f'Results saved to: {output_file}')


Results saved to: TRTR_TSTR_results_air_quality.xlsx
